In [ ]:
# Cài phần đọc ảnh; PyTorch đã có trên Kaggle.
%pip install -q pydicom python-gdcm


In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss

SEED = 42
EPOCHS = 1
MAX_STUDIES = 20  # Chạy thử. Train đầy đủ: đổi thành 0, EPOCHS = 5.
CROP_SIZE = 128
IMAGE_SIZE = 128
BATCH_SIZE = 16
LEARNING_RATE = 0.001
WORKERS = 2
PRETRAINED = False  # Cả ba học từ đầu để so sánh; chỉ thư viện có tùy chọn True.
RUN_ROOT = Path("/kaggle/working/cnn_resnet_smoke_v2")
SPLIT_FILE = RUN_ROOT / "split.csv"

# Tìm ở hai cấp thư mục đầu, tránh quét toàn bộ thư mục ảnh MRI.
input_dir = Path("/kaggle/input")
paths = list(input_dir.glob("*/train_label_coordinates.csv"))
paths += list(input_dir.glob("*/*/train_label_coordinates.csv"))
assert len(paths) == 1, "Hãy gắn đúng một dataset RSNA vào Add Input."
DATA_ROOT = paths[0].parent
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("Thiết bị:", device)
print("Dữ liệu:", DATA_ROOT)
# Khi chuyển sang train đầy đủ, đổi RUN_ROOT sang thư mục mới.
# Các notebook riêng phải cùng seed, MAX_STUDIES và dữ liệu.
# Có thể tải split.csv đã tạo, gắn vào notebook khác rồi sửa SPLIT_FILE để dùng lại.


In [ ]:
import torch
from torch import nn


class BasicBlock(nn.Module):
    """Basic residual block: y = ReLU(F(x) + shortcut(x)).

    Giu dac trung cu bang skip connection, hoc them phan du F(x).
    Vi du: vao 64 kenh -> conv 3x3 giu 64 kenh -> cong truc tiep dau vao.
    """

    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # Shortcut: identity neu cung shape, else conv 1x1 projection.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # Noi tat: cho gradient chay thang, tranh vanishing gradient.
        out = out + self.shortcut(x)
        return self.relu(out)


class ResNet(nn.Module):
    def __init__(self, block, layers, in_channels=1, num_classes=3):
        super().__init__()
        self.in_channels = 64
        # Stem: conv 7x7 + maxpool (chuan ResNet; AdaptiveAvgPool cuoi
        # ho tro moi crop size: 64x64 smoke test hay 128x128/224x224 anh that).
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        layers = [block(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def build_resnet_manual():
    # ResNet-18: 4 stage gom 2, 2, 2, 2 BasicBlock (~11M params).
    # Cau truc tham khao bai bao: https://arxiv.org/abs/1512.03385
    # Ban tu dung day du mot kenh, 3 dau ra, khong chi them mot skip connection.
    return ResNet(BasicBlock, [2, 2, 2, 2], in_channels=1, num_classes=3)


In [ ]:
LABELS = {"Normal/Mild": 0, "Moderate": 1, "Severe": 2}


def read_samples(root, seed, max_studies, split_file=None):
    labels = pd.read_csv(root / "train.csv").melt(
        id_vars="study_id", var_name="target", value_name="severity"
    )
    coords = pd.read_csv(root / "train_label_coordinates.csv")
    coords["target"] = (
        coords.condition.str.lower().str.replace(" ", "_", regex=False)
        + "_" + coords.level.str.lower().str.replace("/", "_", regex=False)
    )
    samples = coords.merge(labels, on=["study_id", "target"], validate="many_to_one")
    missing = samples.severity.isna().sum()
    samples = samples.dropna(subset=["severity"]).copy()
    samples["label"] = samples.severity.map(LABELS)
    if samples.label.isna().any():
        raise ValueError("Unknown severity in train.csv")
    samples["label"] = samples.label.astype(int)
    studies = sorted(samples.study_id.unique())
    if max_studies:
        studies = sorted(random.Random(seed).sample(studies, min(max_studies, len(studies))))
        samples = samples[samples.study_id.isin(studies)].copy()
    if len(studies) < 4:
        raise ValueError("Need at least 4 studies for train/validation/test")
    if split_file is not None and split_file.exists():
        split = pd.read_csv(split_file)
        if set(split.columns) != {"study_id", "split"} or split.study_id.duplicated().any():
            raise ValueError("Split file must contain one study_id and split per study")
        if set(split.study_id) != set(studies) or set(split.split) != {"train", "val", "test"}:
            raise ValueError("Split does not match this dataset/subset. Use the same data and max-studies.")
    else:
        train_ids, remaining_ids = train_test_split(studies, test_size=0.3, random_state=seed)
        val_ids, test_ids = train_test_split(remaining_ids, test_size=0.5, random_state=seed)
        split = pd.DataFrame({"study_id": studies, "split": "train"})
        split.loc[split.study_id.isin(val_ids), "split"] = "val"
        split.loc[split.study_id.isin(test_ids), "split"] = "test"
        if split_file is not None:
            split_file.parent.mkdir(parents=True, exist_ok=True)
            split.to_csv(split_file, index=False)
    samples = samples.merge(split, on="study_id", validate="many_to_one")
    samples["path"] = (
        "train_images/" + samples.study_id.astype(str) + "/"
        + samples.series_id.astype(str) + "/" + samples.instance_number.astype(str) + ".dcm"
    )
    for path in samples.path.unique():
        if not (root / path).is_file():
            raise FileNotFoundError(root / path)
    print(f"Skipped {missing} missing labels; studies per split: {split.split.value_counts().to_dict()}")
    return samples.reset_index(drop=True)


def crop_image(pixels, x, y, size):
    height, width = pixels.shape
    if not (0 <= x < width and 0 <= y < height):
        raise ValueError(f"Coordinate ({x}, {y}) outside image {width}x{height}")
    x, y = min(round(x), width - 1), min(round(y), height - 1)
    # Them vien den de crop o sat mep van dung kich thuoc va tam.
    half = size // 2
    pixels = F.pad(pixels, (half, half, half, half))
    return pixels[y:y + size, x:x + size]


class SpineCrops(Dataset):
    def __init__(self, rows, root, crop_size, image_size):
        self.rows = rows.to_dict("records")
        self.root = root
        self.crop_size = crop_size
        self.image_size = image_size

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        dicom = pydicom.dcmread(self.root / row["path"])
        pixels = torch.tensor(dicom.pixel_array.astype(np.float32))
        pixels = pixels * float(dicom.get("RescaleSlope", 1)) + float(dicom.get("RescaleIntercept", 0))
        pixels = (pixels - pixels.min()) / (pixels.max() - pixels.min()).clamp_min(1e-6)
        if dicom.PhotometricInterpretation == "MONOCHROME1":
            pixels = 1 - pixels
        crop = crop_image(pixels, row["x"], row["y"], self.crop_size)
        image = F.interpolate(crop[None, None], size=(self.image_size, self.image_size),
                              mode="bilinear", align_corners=False)[0]
        return image, row["label"]


def make_model(name, pretrained=False):
    return build_resnet_manual()


def evaluate(model, loader, device):
    model.eval()
    probabilities, targets = [], []
    with torch.inference_mode():
        for images, labels in loader:
            probabilities.append(model(images.to(device)).softmax(1).cpu().numpy())
            targets.extend(labels.tolist())
    probabilities = np.concatenate(probabilities).astype(np.float64)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(1)
    metrics = {
        "log_loss": log_loss(targets, probabilities, labels=[0, 1, 2]),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, labels=[0, 1, 2], average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(targets, predictions, labels=[0, 1, 2]).tolist(),
        "class_counts": np.bincount(targets, minlength=3).tolist(),
        "crops": len(targets),
    }
    return metrics, probabilities


def save_results(output, metrics, probabilities, rows, split_name):
    metrics = dict(metrics, split=split_name)
    (output / f"{split_name}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    predictions = rows.copy()
    predictions[["normal_mild", "moderate", "severe"]] = probabilities
    predictions.to_csv(output / f"{split_name}_predictions.csv", index=False)
    print(json.dumps(metrics, indent=2))



In [ ]:
samples = read_samples(DATA_ROOT, SEED, MAX_STUDIES, SPLIT_FILE)
train_rows = samples[samples.split == "train"]
val_rows = samples[samples.split == "val"]

train_ids = set(train_rows.study_id)
val_ids = set(val_rows.study_id)
test_ids = set(samples.loc[samples.split == "test", "study_id"])
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)
print("Số ca train / validation / test:", len(train_ids), len(val_ids), len(test_ids))

train_data = SpineCrops(train_rows, DATA_ROOT, CROP_SIZE, IMAGE_SIZE)
val_data = SpineCrops(val_rows, DATA_ROOT, CROP_SIZE, IMAGE_SIZE)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=WORKERS)
image, label = train_data[0]
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title(["Normal/Mild", "Moderate", "Severe"][label])
plt.axis("off")
plt.show()
# Tập test chưa được đưa vào model ở bước này.


In [ ]:
def train_model(name):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    output = RUN_ROOT / name
    output.mkdir(parents=True, exist_ok=True)
    if any(output.iterdir()):
        raise FileExistsError(f"Đổi RUN_ROOT để không ghi đè kết quả cũ: {output}")

    pretrained = name == "resnet" and PRETRAINED
    model = make_model(name, pretrained).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    train_loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS,
        generator=torch.Generator().manual_seed(SEED)
    )
    config = {
        "pipeline_version": 2, "model": name, "pretrained": pretrained,
        "seed": SEED, "epochs": EPOCHS, "max_studies": MAX_STUDIES,
        "crop_size": CROP_SIZE, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
        "lr": LEARNING_RATE, "torch_version": str(torch.__version__),
        "parameters": sum(p.numel() for p in model.parameters())
    }
    (output / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    samples.to_csv(output / "samples.csv", index=False)
    history, best_loss = [], float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for step, (images, labels) in enumerate(train_loader, 1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            if step % 100 == 0:
                print(f"{name}: epoch {epoch}, batch {step}/{len(train_loader)}", flush=True)

        # Chỉ validation được dùng để chọn checkpoint trong lúc train.
        metrics, probabilities = evaluate(model, val_loader, device)
        metrics["epoch"] = epoch
        history.append({
            "epoch": epoch, "train_loss": total_loss / len(train_data),
            "val_loss": metrics["log_loss"], "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"]
        })
        pd.DataFrame(history).to_csv(output / "history.csv", index=False)
        print(name, history[-1], flush=True)
        if metrics["log_loss"] < best_loss:
            best_loss = metrics["log_loss"]
            torch.save({"model": model.state_dict(), "config": config, "epoch": epoch},
                       output / "best.pt")
            save_results(output, metrics, probabilities, val_rows, "val")
    return pd.DataFrame(history)


In [ ]:
history_resnet_manual = train_model("resnet_manual")
display(history_resnet_manual)


In [ ]:
# Chỉ chạy khi đã chốt cấu hình và hoàn tất train các model cần so sánh.
# Không dùng điểm test để chọn epoch, learning rate hay kích thước crop.
model_names = ["resnet_manual"]
for name in model_names:
    saved_samples = pd.read_csv(RUN_ROOT / name / "samples.csv")
    pd.testing.assert_frame_equal(saved_samples, samples.reset_index(drop=True), check_dtype=False)

test_rows = samples[samples.split == "test"]
test_data = SpineCrops(test_rows, DATA_ROOT, CROP_SIZE, IMAGE_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=WORKERS)
comparison = []

for name in model_names:
    output = RUN_ROOT / name
    checkpoint = torch.load(output / "best.pt", map_location="cpu", weights_only=True)
    config = checkpoint["config"]
    assert config["crop_size"] == CROP_SIZE and config["image_size"] == IMAGE_SIZE
    model = make_model(config["model"], pretrained=False).to(device)
    model.load_state_dict(checkpoint["model"])
    metrics, probabilities = evaluate(model, test_loader, device)
    metrics["epoch"] = checkpoint["epoch"]
    save_results(output, metrics, probabilities, test_rows, "test")
    comparison.append({
        "model": name, "pretrained": config["pretrained"],
        "epoch": checkpoint["epoch"], "test_log_loss": metrics["log_loss"],
        "test_accuracy": metrics["accuracy"], "test_macro_f1": metrics["macro_f1"]
    })
    del model

comparison = pd.DataFrame(comparison)
comparison.to_csv(RUN_ROOT / "comparison.csv", index=False)
display(comparison)
# Metric tính trên crop có tọa độ cho sẵn, không phải điểm RSNA đầy đủ.
# Chạy thử 20 ca không đủ để kết luận model nào tốt hơn trên dữ liệu thật.
